# RAG Architectures: Which Kind of RAG Do You Need?

Companion notebook for the [RAG Architectures lesson](https://ml-viz-ruby.vercel.app/courses/building-with-llms/14-rag-architectures).

> **Colab:** *File → Save a copy in Drive* before editing, so your work persists.

**The problem.** "We're building RAG" is about as specific as "we're building a
database." The chunk → embed → top-$k$ → generate pipeline is genuinely excellent at
exactly one thing: finding a passage that *says the answer*. It falls apart the moment
the answer isn't sitting in any single passage.

**The core idea.** The **shape of the question** decides the architecture. A question
whose answer spans two documents needs a different retriever than one whose answer is a
`SUM()` over a table. So in this notebook we build **five retrieval architectures over
one shared corpus** and score them against **five query shapes**:

| Shape | Example |
|---|---|
| single-fact | "how many days do I have to get a refund" |
| rare exact token | "what causes ORA-01555" |
| multi-hop | "which customers hit by incident 2431 later churned" |
| corpus-wide | "what do support tickets complain about" |
| aggregation | "total refunds issued in Q3 by region" |

Everything runs locally on NumPy, scikit-learn, and `sqlite3` — no API key, no network.
The stand-ins for LLM calls are clearly marked; the *retrieval* machinery is real.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import re, math, sqlite3
from collections import Counter, defaultdict

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = '#e2e8f0'
plt.rcParams['axes.labelcolor'] = '#e2e8f0'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#334155'
plt.rcParams['grid.color'] = '#1e293b'
plt.rcParams['figure.figsize'] = (9, 5)
np.random.seed(0)

## 1. From scratch — one corpus, five architectures

### The corpus

A miniature knowledge base for a fictional company, *Northwind Cloud*. It is deliberately
built so that each query shape needs something different:

- **policy** — the refund window lives in exactly one chunk (*single-fact*).
- **incident reports** — contain a rare literal, `ORA-01555` (*rare exact token*), and a
  chunk (`i7`) that is meaningless without its document context.
- **accounts** — churn records that never mention the incident, so linking "who was
  affected" to "who churned" requires **two hops** across documents.
- **tickets** — 12 tickets over 4 themes with a skewed distribution (6 billing, 3 auth,
  2 performance, 1 status). No top-$k$ slice can report the *distribution* (*corpus-wide*).

In [ ]:
# (id, source document, theme-or-None, text)
CORPUS = [
    ("p1", "policy", None, "A refund is available within 30 days of purchase for any Northwind Cloud plan."),
    ("p2", "policy", None, "To request a refund, email billing support with the invoice number."),
    ("p3", "policy", None, "Enterprise contracts are refunded pro rata rather than in full."),
    ("p4", "policy", None, "Service credits are issued automatically when monthly uptime falls below 99.9 percent."),
    ("i1", "incident-2431", None, "Incident 2431 began on March 14 when the Vega storage cluster lost quorum."),
    ("i2", "incident-2431", None, "During incident 2431 customers Acme Corp and Bolt Industries reported write failures."),
    ("i3", "incident-2431", None, "The root cause of incident 2431 was an undersized undo tablespace raising ORA-01555."),
    ("i4", "incident-2431", None, "ORA-01555 is a snapshot too old error raised when undo data needed by a long query is overwritten."),
    ("i7", "incident-2431", None, "The fix was to resize the tablespace and restart the affected nodes."),
    ("i5", "incident-2477", None, "Incident 2477 in April degraded the Lyra API gateway for forty minutes."),
    ("i6", "incident-2477", None, "Bolt Industries and Cinder Labs were the customers affected by incident 2477."),
    ("c1", "accounts", None, "Acme Corp churned in May citing repeated reliability problems."),
    ("c2", "accounts", None, "Bolt Industries churned in June after escalating two severity one incidents."),
    ("c3", "accounts", None, "Cinder Labs renewed in July and expanded to the enterprise plan."),
    ("c4", "accounts", None, "Delta Freight renewed in June with no open escalations."),
    ("d1", "runbook", None, "The Vega storage cluster runbook covers quorum loss and manual failover steps."),
    ("d2", "runbook", None, "The Lyra API gateway runbook covers rate limit tuning and connection draining."),
    ("d3", "runbook", None, "Undo tablespace sizing guidance recommends monitoring retention against longest query time."),
    # --- support tickets: 6 billing, 3 auth, 2 performance, 1 status ---
    ("t1", "tickets", "billing", "Support ticket about a duplicate charge on the monthly invoice."),
    ("t2", "tickets", "billing", "Support ticket asking why the invoice total changed after a plan upgrade."),
    ("t3", "tickets", "billing", "Support ticket disputing a charge already covered by a service credit."),
    ("t4", "tickets", "billing", "Support ticket requesting an invoice reissued to a new billing address."),
    ("t5", "tickets", "billing", "Support ticket about a failed payment method on the monthly invoice."),
    ("t6", "tickets", "billing", "Support ticket asking to split one invoice across two cost centres."),
    ("t7", "tickets", "auth", "Support ticket about rotating API keys without downtime."),
    ("t8", "tickets", "auth", "Support ticket about single sign on failing after an identity provider change."),
    ("t9", "tickets", "auth", "Support ticket about expired service account tokens blocking deploys."),
    ("t10", "tickets", "performance", "Support ticket about slow query alerts during nightly batch jobs."),
    ("t11", "tickets", "performance", "Support ticket about elevated latency on the Lyra API gateway."),
    ("t12", "tickets", "status", "Support ticket about unclear status page updates during an outage."),
]
DOC_IDS = [d[0] for d in CORPUS]
SOURCES = [d[1] for d in CORPUS]
THEMES  = [d[2] for d in CORPUS]
TEXTS   = [d[3] for d in CORPUS]
IDX = {d: i for i, d in enumerate(DOC_IDS)}

DOC_SUMMARY = {
    "policy": "Northwind Cloud billing and refund policy.",
    "incident-2431": "Incident 2431, the March 14 Vega storage cluster outage.",
    "incident-2477": "Incident 2477, the April Lyra API gateway degradation.",
    "accounts": "Customer account renewal and churn records.",
    "tickets": "Support ticket archive.",
    "runbook": "Operational runbooks for Northwind Cloud services.",
}

QUERIES = {
    "single-fact":  "how many days do I have to get a refund",
    "exact-token":  "what causes ORA-01555",
    "multi-hop":    "which customers hit by incident 2431 later churned",
    "corpus-wide":  "what do support tickets complain about",
    "aggregation":  "total refunds issued in Q3 by region",
}
print(f"{len(CORPUS)} chunks / {len(set(SOURCES))} documents / "
      f"{len([t for t in THEMES if t])} tickets in {len({t for t in THEMES if t})} themes")

### The sparse representation: TF-IDF

Everything else is built on top of this. Each chunk becomes a vector over the corpus
vocabulary, where term $t$ in document $d$ gets weight

$$
w_{t,d} = \underbrace{\mathrm{tf}(t, d)}_{\text{count in } d} \cdot
          \underbrace{\left(\ln\frac{1+N}{1+\mathrm{df}(t)} + 1\right)}_{\text{idf: rarity}}
$$

then the vector is L2-normalised so a dot product *is* cosine similarity. Note the
vocabulary is built from the corpus only — an index knows nothing about words it has
never seen, and unknown query terms are simply dropped.

**Shapes:** `tfidf(text) -> (V,)`, `TFIDF_DOCS -> (n_chunks, V)`.

In [ ]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

VOCAB = sorted({t for x in TEXTS for t in tokenize(x)})   # the index knows only its corpus
V_INDEX = {t: i for i, t in enumerate(VOCAB)}

N = len(TEXTS)
DF = Counter()
for x in TEXTS:
    for t in set(tokenize(x)):
        DF[t] += 1
IDF = np.array([math.log((1 + N) / (1 + DF[t])) + 1 for t in VOCAB])

def tfidf(text):
    v = np.zeros(len(VOCAB))
    for t in tokenize(text):
        if t in V_INDEX:
            v[V_INDEX[t]] += 1.0
    v *= IDF
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

TFIDF_DOCS = np.vstack([tfidf(x) for x in TEXTS])
print("sparse matrix:", TFIDF_DOCS.shape)

## 2. The library way — cross-checking the sparse stage

`TfidfVectorizer` computes exactly this (same smoothing, same L2 normalisation), so our
scores should agree to floating-point noise. If they didn't, everything downstream would
be measuring our bug rather than the architecture.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

vec = TfidfVectorizer(token_pattern=r"[a-z0-9]+", lowercase=True)
X = vec.fit_transform(TEXTS)

q_demo = QUERIES["single-fact"]
sk = cosine_similarity(vec.transform([q_demo]), X).ravel()
ours = TFIDF_DOCS @ tfidf(q_demo)
print("max abs diff vs sklearn:", np.abs(sk - ours).max())
assert np.allclose(sk, ours, atol=1e-8)
print("from-scratch TF-IDF matches scikit-learn")

### The dense representation: a low-rank projection

TF-IDF is *lexical* — it cannot match "get my money back" to "refund policy". A real
embedding model is dense and semantic. We approximate one by projecting TF-IDF onto its
top-$r$ singular directions (**LSA**):

$$
\mathbf{X} \approx \mathbf{U}_r \mathbf{\Sigma}_r \mathbf{V}_r^\top,
\qquad
\text{embed}(q) = \frac{\mathrm{tfidf}(q)\,\mathbf{V}_r}{\lVert \cdot \rVert}
$$

This is not a toy detail — it reproduces **the exact trade a real embedding model makes**.
Low rank is what makes the space semantic (paraphrases collapse onto the same directions)
and it is *the same mechanism* that loses rare literal tokens: a term that appears in two
chunks out of thousands doesn't survive compression into a few hundred dimensions.

We use an aggressive $r = 5$ because the corpus is tiny; a production model compresses far
less aggressively but faces the same effect at corpus scale.

In [ ]:
RANK = 5
U, S, Vt = np.linalg.svd(TFIDF_DOCS, full_matrices=False)
BASIS = Vt[:RANK].T                       # (V, RANK)

def embed(text):
    """Dense 'embedding': TF-IDF projected onto RANK latent dimensions."""
    v = tfidf(text) @ BASIS
    n = np.linalg.norm(v)
    return v / n if n > 0 else v

DOC_VECS = np.vstack([embed(x) for x in TEXTS])
print("dense matrix:", DOC_VECS.shape,
      "| variance kept:", round(float((S[:RANK]**2).sum() / (S**2).sum()), 3))

# cross-check against sklearn's TruncatedSVD (cosine is invariant to the per-component
# sign flips the two solvers can disagree on, so we compare similarities, not vectors)
from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(n_components=RANK, algorithm="arpack", random_state=0).fit(X)

def sk_embed(t):
    v = svd.transform(vec.transform([t])).ravel()
    return v / (np.linalg.norm(v) or 1)

sk_docs = np.vstack([sk_embed(t) for t in TEXTS])
diff = np.abs((sk_docs @ sk_embed(q_demo)) - (DOC_VECS @ embed(q_demo))).max()
print("cosine agreement with sklearn TruncatedSVD:", float(diff))
assert diff < 1e-8

### Architecture 1 — naive vector RAG

`chunk → embed → top-k → generate`. The whole thing is one matrix-vector product and an
`argsort`. This is the baseline every other architecture has to beat.

In [ ]:
def naive_dense(query, k=4, doc_vecs=None):
    dv = DOC_VECS if doc_vecs is None else doc_vecs
    scores = dv @ embed(query)
    return [(DOC_IDS[i], round(float(scores[i]), 3)) for i in np.argsort(-scores)[:k]]

def show(pairs):
    for did, s in pairs:
        print(f"  {did:>3}  {s:>6.3f}  {TEXTS[IDX[did]][:66]}")

print("Q:", QUERIES["single-fact"]); show(naive_dense(QUERIES["single-fact"], k=3))
print()
print("Q:", QUERIES["exact-token"]); show(naive_dense(QUERIES["exact-token"], k=3))

**What to notice.** Two failures are already visible, and both are characteristic rather
than accidental:

- On the **refund** query the top hit is `p2` ("*how to request* a refund") rather than
  `p1` ("*within 30 days*"). Both are about refunds; the embedding cannot tell which one
  answers *this* question. That is what a reranker is for.
- On **`ORA-01555`** the top hit is `d3` — a chunk about tablespace sizing that never
  mentions the error code. The rare literal has been smeared into its semantic
  neighbourhood by the projection. That is what BM25 is for.

### Architecture 2 — hybrid retrieval + reranking

**Step 1: BM25.** The lexical retriever, scoring literal term overlap with saturation
($k_1$) and length normalisation ($b$):

$$
\text{BM25}(q, d) = \sum_{t \in q} \text{idf}(t) \cdot
\frac{f_{t,d} \, (k_1 + 1)}{f_{t,d} + k_1\left(1 - b + b\frac{|d|}{\text{avgdl}}\right)}
$$

In [ ]:
class BM25:
    def __init__(self, texts, k1=1.5, b=0.75):
        self.docs = [tokenize(t) for t in texts]
        self.k1, self.b = k1, b
        self.avgdl = sum(len(d) for d in self.docs) / len(self.docs)
        df = Counter()
        for d in self.docs:
            for t in set(d):
                df[t] += 1
        n = len(self.docs)
        self.idf = {t: math.log(1 + (n - c + 0.5) / (c + 0.5)) for t, c in df.items()}
        self.tf = [Counter(d) for d in self.docs]

    def scores(self, query):
        out = np.zeros(len(self.docs))
        for t in tokenize(query):
            if t not in self.idf:
                continue
            for i, tf in enumerate(self.tf):
                f = tf.get(t, 0)
                if f:
                    dl = len(self.docs[i])
                    out[i] += self.idf[t] * f * (self.k1 + 1) / (
                        f + self.k1 * (1 - self.b + self.b * dl / self.avgdl))
        return out

bm25 = BM25(TEXTS)
bs = bm25.scores(QUERIES["exact-token"])
print("BM25 on the query dense retrieval just failed:")
for i in np.argsort(-bs)[:3]:
    print(f"  {DOC_IDS[i]:>3}  {bs[i]:>6.3f}  {TEXTS[i][:66]}")

**Step 2: fuse, then rerank.** BM25 scores and cosine similarities are on incomparable
scales, so we fuse **ranks** instead of scores with Reciprocal Rank Fusion:

$$
\text{RRF}(d) = \sum_{r} \frac{1}{k + \text{rank}_r(d)}, \qquad k \approx 60
$$

Then a **cross-encoder** re-scores the shortlist. The distinction that matters: the
first stage is a *bi-encoder* — query and document are embedded **independently**, which
is what lets you precompute the index and is also why it is blunt. A cross-encoder reads
the pair **together**, so it can see term overlap the bi-encoder structurally cannot.
Too slow for a corpus, ideal for 10–150 candidates.

In [ ]:
def rrf(rank_lists, k=60):
    fused = defaultdict(float)
    for ranks in rank_lists:
        for pos, i in enumerate(ranks):
            fused[i] += 1.0 / (k + pos + 1)
    return sorted(fused, key=lambda i: -fused[i])

def cross_encoder(query, i):
    """Stand-in for a cross-encoder: scores the (query, doc) PAIR jointly.
    Rewards covering the query's terms inside a short passage."""
    qt = set(tokenize(query))
    dt = tokenize(TEXTS[i])
    if not qt:
        return 0.0
    return (len(qt & set(dt)) / len(qt)) * (1.0 / (1.0 + 0.02 * len(dt)))

def hybrid_rerank(query, k=4, shortlist=10):
    dense_rank  = list(np.argsort(-(DOC_VECS @ embed(query))))[:shortlist]
    sparse_rank = list(np.argsort(-bm25.scores(query)))[:shortlist]
    fused = rrf([dense_rank, sparse_rank])[:shortlist]
    order = sorted(fused, key=lambda i: -cross_encoder(query, i))
    return [(DOC_IDS[i], round(cross_encoder(query, i), 3)) for i in order[:k]]

print("hybrid + rerank:"); show(hybrid_rerank(QUERIES["exact-token"], k=3))

# top-1 is where the two pipelines differ most
GOLD1 = {"single-fact": {"p1"}, "exact-token": {"i3", "i4"},
         "multi-hop": {"i2"}, "corpus-wide": {f"t{i}" for i in range(1, 13)}}
print(f"\n{'query shape':>14} {'dense@1':>9} {'hybrid@1':>9}   correct?")
for shape, gold in GOLD1.items():
    d1 = naive_dense(QUERIES[shape], k=1)[0][0]
    h1 = hybrid_rerank(QUERIES[shape], k=1)[0][0]
    print(f"{shape:>14} {d1:>9} {h1:>9}   dense={str(d1 in gold):>5} hybrid={h1 in gold}")

**What to notice.** On `exact-token`, dense retrieval's top hit is wrong (`d3`) and hybrid's
is right (`i3`) — the sparse half recovered a literal the dense half had dissolved. On
`single-fact` **both** still get it wrong at rank 1, which is the honest result: fusion
fixes lexical misses, not "which of two on-topic passages actually answers the question."

### Contextual retrieval

Chunk `i7` reads *"The fix was to resize the tablespace and restart the affected nodes."*
Which fix? Which incident? Embedded alone, it is nearly unretrievable. **Contextual
retrieval** prepends a short situating blurb — in production written by an LLM that has
seen the whole document — *before* embedding. Anthropic reports this cuts top-20 retrieval
failures by 49%, and 67% when combined with reranking.

In [ ]:
CONTEXTUAL = [f"{DOC_SUMMARY[s]} {t}" for s, t in zip(SOURCES, TEXTS)]
CTX_VECS = np.vstack([embed(x) for x in CONTEXTUAL])
print("i7 as indexed, with context:\n ", CONTEXTUAL[IDX["i7"]], "\n")

for q_ctx in ["what remediation was applied to the Vega cluster",
              "how was the March 14 problem fixed",
              "what did we do about the Vega storage outage"]:
    print("Q:", q_ctx)
    print("  plain      :", [d for d, _ in naive_dense(q_ctx, k=3)])
    print("  contextual :", [d for d, _ in naive_dense(q_ctx, k=3, doc_vecs=CTX_VECS)])

**What to notice.** Three outcomes, all worth seeing: on the first query context changes
nothing (the chunk was already findable), on the second it promotes `i7` from rank 3 to
rank 1, and on the third it **recovers a chunk plain retrieval missed entirely**. That
last case is the one the 49% figure is made of — and it costs nothing at query time,
because the work happened at indexing time.

### Architecture 3 — GraphRAG

Two things top-$k$ structurally cannot do: **join** facts across documents, and describe
a property of the **whole corpus**. GraphRAG builds an index that can.

**Indexing** (all offline, and in production every step below is an LLM call):
1. extract entities and relations from each chunk,
2. assemble the co-occurrence graph,
3. detect communities and write a **community report** for each.

**Querying** splits in two, and picking the wrong mode is the classic GraphRAG mistake:
- **local search** — seed from entities in the question, fan out to neighbours. For
  "who was involved and what happened."
- **global search** — map-reduce over community *reports*, never over raw chunks. For
  "what are the recurring themes." The report layer is what materialises corpus-level
  knowledge that no retrieval over chunks can produce.

In [ ]:
ENTITIES = ["Acme Corp", "Bolt Industries", "Cinder Labs", "Delta Freight",
            "Vega", "Lyra", "incident 2431", "incident 2477", "ORA-01555"]

def extract_entities(text):
    """Stand-in for LLM entity extraction (a dictionary match here)."""
    return [e for e in ENTITIES if e.lower() in text.lower()]

ENT_TO_CHUNKS = defaultdict(set)
CHUNK_TO_ENTS = {}
for i, t in enumerate(TEXTS):
    ents = extract_entities(t)
    CHUNK_TO_ENTS[i] = ents
    for e in ents:
        ENT_TO_CHUNKS[e].add(i)

EDGES = Counter()
for i, ents in CHUNK_TO_ENTS.items():
    for a in range(len(ents)):
        for b in range(a + 1, len(ents)):
            EDGES[tuple(sorted((ents[a], ents[b])))] += 1

print("graph:", len(ENT_TO_CHUNKS), "entities,", len(EDGES), "edges")
for (a, b), w in EDGES.most_common(5):
    print(f"   {a} — {b}  (x{w})")

In [ ]:
def local_search(query, hops=1, k=5):
    frontier = set(extract_entities(query))
    for _ in range(hops):
        nxt = set(frontier)
        for (a, b) in EDGES:
            if a in frontier: nxt.add(b)
            if b in frontier: nxt.add(a)
        frontier = nxt
    cand = set()
    for e in frontier:
        cand |= ENT_TO_CHUNKS[e]
    return [DOC_IDS[i] for i in sorted(cand, key=lambda i: -(DOC_VECS[i] @ embed(query)))[:k]]

COMMUNITIES = defaultdict(list)
for i, s in enumerate(SOURCES):
    COMMUNITIES[s].append(i)

def community_report(name):
    """Stand-in for the LLM-written community report."""
    idxs = COMMUNITIES[name]
    themes = Counter(THEMES[i] for i in idxs if THEMES[i])
    body = DOC_SUMMARY[name]
    if themes:
        body += " Recurring themes: " + ", ".join(f"{t} ({c} tickets)" for t, c in themes.most_common())
        body += ". " + " ".join(TEXTS[i] for i in idxs[:2])
    else:
        body += " " + " ".join(TEXTS[i] for i in idxs)
    return {"community": name, "size": len(idxs), "themes": sorted(themes), "summary": body}

REPORTS = [community_report(c) for c in COMMUNITIES]

def global_search(query, top=1):
    qv = embed(query)
    return sorted(REPORTS, key=lambda r: -(embed(r["summary"]) @ qv))[:top]

print("local search  :", local_search(QUERIES["multi-hop"]))
gr = global_search(QUERIES["corpus-wide"])[0]
print("global search :", gr["community"])
print("  report ->", gr["summary"][:150])

**What to notice.** Local search returns `i2` (who was affected) **and** `c1`/`c2` (who
churned) — chunks from two different documents that share no vocabulary. The graph edge
did the join that cosine similarity could not.

Global search returns *one passage* that names all four ticket themes **with their
counts**. Retrieving the top 5 of 12 tickets could never produce that distribution — and
the report is smaller than the chunks it summarises, so it is cheaper to send too.

### Architecture 4 — agentic RAG

The pipelines above retrieve **once**. An agent retrieves, **grades what came back**, and
retrieves again with a better query — the ReAct loop with search as the action. This is
the only family that handles questions where you cannot know what to search for until you
have seen the first result.

Our stand-in grader asks what a real grader asks — *is anything still missing?* — by
checking that every customer named in the evidence also has a known account outcome.

In [ ]:
CUSTOMERS = ["Acme Corp", "Bolt Industries", "Cinder Labs", "Delta Freight"]

def grade(query, seen_idx):
    """Stand-in for an LLM grader: is the accumulated evidence sufficient?"""
    named = {e for i in seen_idx for e in CHUNK_TO_ENTS[i] if e in CUSTOMERS}
    if named:
        resolved = {e for e in named
                    if any(SOURCES[i] == "accounts" and e in CHUNK_TO_ENTS[i] for i in seen_idx)}
        return len(resolved) / len(named)
    stop = {"what", "do", "the", "a", "in", "of", "is", "about", "are", "to", "have", "i"}
    need = set(tokenize(query)) - stop
    got = set()
    for i in seen_idx:
        got |= set(tokenize(TEXTS[i]))
    return len(need & got) / max(len(need), 1)

def agentic_rag(query, max_rounds=4, k=3, verbose=True):
    """retrieve -> grade -> reformulate -> retrieve again. Seen chunks are masked
    out, so each round genuinely explores new evidence."""
    seen, calls, sub = [], 0, query
    for r in range(max_rounds):
        calls += 1                          # the retrieval-planning call
        scores = DOC_VECS @ embed(sub)
        scores[seen] = -np.inf              # don't re-read what we already have
        new = [int(i) for i in np.argsort(-scores)[:k]]
        seen += new
        cov = grade(query, seen)
        if verbose:
            print(f"  round {r+1}: {sub[:56]!r}")
            print(f"           -> {[DOC_IDS[i] for i in new]}  sufficiency={cov:.2f}")
        if cov >= 0.99:
            break
        calls += 1                          # the reformulation call
        gaps = sorted({e for i in seen for e in CHUNK_TO_ENTS[i] if e in CUSTOMERS})
        sub = query + " " + (" ".join(gaps) if gaps else "other themes")
    return [DOC_IDS[i] for i in seen], calls

print("one-shot retrieval:", [d for d, _ in naive_dense(QUERIES["multi-hop"], k=3)])
print("\nagentic:")
hits, calls = agentic_rag(QUERIES["multi-hop"])
print("  evidence:", hits, "| LLM calls:", calls)

**What to notice.** Round 1 retrieves the incident chunks and the grader returns
**sufficiency 0.00** — two customers named, neither with a known outcome. The agent
rewrites the query with those entity names and round 2 pulls the account records. Same
index, same embedder as architecture 1; the loop is the entire difference.

The cost is right there in the output: **3 LLM calls** where naive RAG made 0, plus more
context tokens. Multiply by your query volume before adopting this by default.

### Architecture 5 — structured RAG (text-to-SQL)

"Total refunds in Q3 by region" has no answer in any passage. Embeddings measure semantic
similarity; they cannot count, sum, or group. The right retrieval here is **against a
database**: retrieve the *schema*, generate SQL, execute it, narrate the result.

In [ ]:
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE refunds (id INTEGER PRIMARY KEY, region TEXT, quarter TEXT, amount REAL);
INSERT INTO refunds (region, quarter, amount) VALUES
 ('EMEA','Q3',1200.0),('EMEA','Q3',800.0),('AMER','Q3',2500.0),
 ('AMER','Q3',1500.0),('APAC','Q3',600.0),('EMEA','Q2',400.0);
""")

# step 1: retrieve the schema (this is the "retrieval" in structured RAG)
SCHEMA = "\n".join(r[0] for r in con.execute("SELECT sql FROM sqlite_master WHERE type='table'"))
print("retrieved schema:\n", SCHEMA, "\n")

# step 2: generate SQL conditioned on it (stand-in for the LLM planner)
def text_to_sql(question):
    q = question.lower()
    if "refund" in q and "region" in q:
        return ("SELECT region, SUM(amount) AS total FROM refunds "
                "WHERE quarter='Q3' GROUP BY region ORDER BY total DESC")
    return "SELECT COUNT(*) FROM refunds"

sql = text_to_sql(QUERIES["aggregation"])
print("generated SQL:", sql)
print("result       :", con.execute(sql).fetchall())
print("\nwhat vector retrieval returns for the same question:",
      [d for d, _ in naive_dense(QUERIES["aggregation"], k=3)])

**What to notice.** The SQL path returns the exact figures. Vector retrieval returns three
chunks about accounts and policies — plausible-looking context from which a generator would
confidently invent a number. This is the failure mode that makes structured routing
non-optional once anyone asks a quantitative question.

## 3. Visualise it — one scoreboard, five architectures

Now score all five architectures on all five query shapes. The metric is deliberately
**not** chunk recall: GraphRAG's global search returns a *report*, and text-to-SQL returns
*rows*, so counting retrieved chunk ids would be meaningless across architectures.

Instead we ask the question an evaluator actually cares about — **is the answer derivable
from the evidence this architecture put in the prompt?** — by checking for the required
facts. That is `context_recall` in Ragas terms, and it is comparable across every
architecture because it only looks at the evidence text.

In [ ]:
REQUIRED = {
    "single-fact": ["30 days"],
    "exact-token": ["undo tablespace", "snapshot too old"],
    "multi-hop":   ["Acme Corp and Bolt Industries", "Acme Corp churned", "Bolt Industries churned"],
    "corpus-wide": ["billing", "auth", "performance", "status"],
    "aggregation": ["AMER 4000", "EMEA 2000", "APAC 600"],
}

def coverage(evidence_text, shape):
    facts = REQUIRED[shape]
    return sum(1 for f in facts if f.lower() in evidence_text.lower()) / len(facts)

K = 5

def ev_naive(q):   return [TEXTS[IDX[d]] for d, _ in naive_dense(q, k=K)]
def ev_hybrid(q):  return [TEXTS[IDX[d]] for d, _ in hybrid_rerank(q, k=K)]
def ev_graph(q):
    if extract_entities(q):
        return [TEXTS[IDX[d]] for d in local_search(q, hops=1, k=K)]
    return [r["summary"] for r in global_search(q, top=1)]
def ev_agentic(q): return [TEXTS[IDX[d]] for d in agentic_rag(q, verbose=False)[0]]
def ev_sql(q):
    try:
        rows = con.execute(text_to_sql(q)).fetchall()
    except sqlite3.Error:
        return []
    return [" ".join(f"{r[0]} {r[1]:.0f}" for r in rows)] if rows and len(rows[0]) == 2 else []

ARCHS = {"naive dense": ev_naive, "hybrid+rerank": ev_hybrid,
         "graphRAG": ev_graph, "agentic": ev_agentic, "text-to-SQL": ev_sql}

shapes = list(QUERIES)
M   = np.zeros((len(ARCHS), len(shapes)))
TOK = np.zeros((len(ARCHS), len(shapes)))
for r, (aname, fn) in enumerate(ARCHS.items()):
    for c, shape in enumerate(shapes):
        text = "\n".join(fn(QUERIES[shape]))
        M[r, c]   = coverage(text, shape)
        TOK[r, c] = len(tokenize(text))

print(" " * 15 + "".join(f"{s:>14}" for s in shapes))
for r, a in enumerate(ARCHS):
    print(f"{a:>14} " + "".join(f"{M[r,c]:>14.2f}" for c in range(len(shapes))))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.4))

im = axes[0].imshow(M, cmap="viridis", vmin=0, vmax=1, aspect="auto")
axes[0].set_xticks(range(len(shapes)), shapes, rotation=18, ha="right")
axes[0].set_yticks(range(len(ARCHS)), list(ARCHS))
for r in range(M.shape[0]):
    for c in range(M.shape[1]):
        axes[0].text(c, r, f"{M[r,c]:.2f}", ha="center", va="center",
                     color="#0f1117" if M[r, c] > 0.55 else "#e2e8f0", fontsize=9)
axes[0].set_title("Is the answer derivable from the evidence?")
axes[0].grid(False)
fig.colorbar(im, ax=axes[0], shrink=0.85)

axes[1].bar(range(len(ARCHS)), TOK.mean(axis=1), color="#6366f1")
axes[1].set_xticks(range(len(ARCHS)), list(ARCHS), rotation=18, ha="right")
axes[1].set_ylabel("mean context tokens per query")
axes[1].set_title("What it costs to send that evidence")
axes[1].grid(axis="y")
plt.tight_layout()
plt.show()

**What to notice — this table is the whole lesson.**

- **No row wins everywhere, and no column is won by the same architecture.** The
  "best RAG" question is malformed.
- Naive and hybrid are **equal on coverage** here for the shapes where both find the
  evidence — hybrid's win showed up earlier at *rank 1*, which an @5 metric hides. Pick
  your $k$ to match how much context you actually send.
- **GraphRAG is the only architecture that scores 1.00 on `corpus-wide`**, and it does so
  while sending the *fewest* tokens, because a community report is a compression, not a
  sample.
- **Agentic matches GraphRAG on `multi-hop`** with no special index — but look at the
  token bar: roughly 2–3× the context of every one-shot pipeline, plus the LLM calls the
  bar doesn't show.
- **Every text architecture scores 0.00 on `aggregation`.** Only the SQL path answers it,
  in 6 tokens.

## 4. Tradeoffs & when to use which

| Architecture | Best for | Weak at | Query latency | Indexing cost |
|---|---|---|---|---|
| **Naive vector** | single-fact lookup on homogeneous prose | rare literals, joins, synthesis | ~1 s | one embedding pass |
| **Hybrid + rerank** | exact tokens, precision at rank 1 | joins, synthesis | ~1.5–2 s | embeddings + inverted index (+ LLM per chunk if contextual) |
| **Hierarchical (RAPTOR)** | "summarise this document" over long structured docs | cross-document joins | ~1.5 s | LLM summary per tree node |
| **GraphRAG** | multi-hop joins, corpus-wide themes | fast-changing corpora | ~3–4 s | **an LLM call per chunk + per community** |
| **Self-correcting (CRAG)** | stopping confident answers from bad context | doesn't improve retrieval itself | +1 grader call | none |
| **Agentic** | open-ended research, unknown-unknowns | cost, latency, determinism | 5–15 s | none |
| **Multimodal (ColPali)** | charts, tables, scanned pages | text-only corpora (wasteful) | ~2 s | hundreds of vectors per page |
| **Text-to-SQL** | counts, sums, rankings | anything unstructured | ~2 s | index the schema, not the rows |

### Complexity

For $n$ chunks, $d$ embedding dimensions, $k$ retrieved:

- **Indexing** — naive $O(nd)$; contextual adds $O(n)$ LLM calls; GraphRAG adds $O(n)$
  extraction calls **plus** $O(\#\text{communities})$ summarisation calls.
- **Query** — exact search $O(nd)$, ANN (HNSW/IVF) $O(d \log n)$; reranking adds
  $O(m)$ cross-encoder passes over an $m$-candidate shortlist; agentic multiplies the
  whole thing by the number of rounds.
- **Storage** — one vector per chunk, except late-interaction multimodal, which stores
  hundreds per page.

### Failure modes to watch

1. **Retrieval miss** — the answer was never in the top-$k$. No prompt fixes this.
2. **Rare-literal smearing** — the dense-only failure demonstrated above.
3. **Context-free chunks** — chunks that lost their document framing; contextual retrieval
   is the cheap fix.
4. **Sampling bias on synthesis** — top-$k$ over a skewed corpus reports the *majority*
   theme as if it were the whole picture.
5. **Silent numeric hallucination** — plausible chunks, invented arithmetic.
6. **Stale index** — GraphRAG's graph and community reports go stale fastest and cost the
   most to rebuild.
7. **Agent loops that never converge** — always cap the rounds.

## 5. ✏️ Your turn

Three exercises. Each has a code outline with `# TODO(you)` blanks, an assertion cell that
passes silently when correct, and a collapsed solution.

### Exercise 1 — the router

The highest-leverage component in most production RAG systems isn't a retriever, it's the
**router** that decides which retriever to use. Write `adaptive_route(query)` returning one
of `"text-to-SQL"`, `"graphRAG-global"`, `"agentic"`, or `"hybrid+rerank"` (the default).

Watch the trap: `"how many days do I have to get a refund"` contains *"how many"* but is
**not** an aggregation query. Rules that match on surface words alone will misroute it —
which is exactly why production routers are usually a classifier or an LLM, not a keyword list.

In [ ]:
def adaptive_route(query):
    q = query.lower()
    # TODO(you): aggregate intent -> "text-to-SQL"
    #   careful: "how many days" is NOT an aggregation over records
    # TODO(you): corpus-wide synthesis -> "graphRAG-global"
    # TODO(you): named entity + a linking word ("later", "also", " and ") -> "agentic"
    # TODO(you): otherwise -> "hybrid+rerank"
    return "hybrid+rerank"

for shape, qq in QUERIES.items():
    print(f"{adaptive_route(qq):>16}  <-  [{shape}] {qq}")

In [ ]:
assert adaptive_route(QUERIES["aggregation"])  == "text-to-SQL"
assert adaptive_route(QUERIES["corpus-wide"])  == "graphRAG-global"
assert adaptive_route(QUERIES["multi-hop"])    == "agentic"
assert adaptive_route(QUERIES["exact-token"])  == "hybrid+rerank"
assert adaptive_route(QUERIES["single-fact"])  == "hybrid+rerank"   # the trap
print("router routes all five shapes correctly")

<details>
<summary>Solution</summary>

```python
def adaptive_route(query):
    q = query.lower()
    if any(w in q for w in ("total ", "average ", "sum of", "count of", "by region", "per region")):
        return "text-to-SQL"
    if any(w in q for w in ("complain about", "themes", "overall", "across all", "summarise", "summarize")):
        return "graphRAG-global"
    if extract_entities(query) and any(w in q for w in ("later", "also", " and ")):
        return "agentic"
    return "hybrid+rerank"
```

The aggregation rule matches `"total "` and `"by region"` rather than `"how many"`,
which is what keeps the single-fact query out of the SQL path.
</details>

### Exercise 2 — precision and recall pull in opposite directions

Cranking $k$ up "to be safe" is the most common RAG mistake. Implement `context_precision`
(what fraction of retrieved chunks are actually relevant) and watch it collapse as recall
saturates — every irrelevant chunk is noise the generator must ignore, and tokens you pay for.

In [ ]:
GOLD_EXACT = {"i3", "i4"}      # the two chunks that answer "what causes ORA-01555"

def context_precision(retrieved_ids, gold):
    # TODO(you): fraction of retrieved ids that are in gold (0.0 if nothing retrieved)
    return 0.0

def context_recall(retrieved_ids, gold):
    # TODO(you): fraction of gold ids that were retrieved
    return 0.0

for kk in (1, 2, 5, 10):
    ids = [d for d, _ in hybrid_rerank(QUERIES["exact-token"], k=kk)]
    print(f"k={kk:>2}  recall={context_recall(ids, GOLD_EXACT):.2f}  "
          f"precision={context_precision(ids, GOLD_EXACT):.2f}")

In [ ]:
ids2 = [d for d, _ in hybrid_rerank(QUERIES["exact-token"], k=2)]
ids10 = [d for d, _ in hybrid_rerank(QUERIES["exact-token"], k=10)]
assert context_recall(ids2, GOLD_EXACT) == 1.0
assert context_precision(ids2, GOLD_EXACT) == 1.0
assert context_recall(ids10, GOLD_EXACT) == 1.0        # recall saturated at k=2
assert context_precision(ids10, GOLD_EXACT) == 0.2     # ...and precision fell 5x
print("k=2 is the sweet spot here; k=10 adds 8 chunks of pure noise")

<details>
<summary>Solution</summary>

```python
def context_precision(retrieved_ids, gold):
    return len(set(retrieved_ids) & gold) / len(retrieved_ids) if retrieved_ids else 0.0

def context_recall(retrieved_ids, gold):
    return len(set(retrieved_ids) & gold) / len(gold) if gold else 0.0
```

Recall hits 1.0 at $k=2$ and never improves; precision falls from 1.0 to 0.2 by $k=10$.
Everything past $k=2$ is cost and distraction. This is why you tune $k$ against a labelled
set instead of picking a round number.
</details>

### Exercise 3 — a corrective-RAG gate

**CRAG** puts a grader between retrieval and generation and branches three ways:
**correct** (use the context), **ambiguous** (use it *and* fall back to another source),
**incorrect** (discard it — never generate from it). Implement the gate.

Use the cross-encoder score of the best candidate as the grader signal, with thresholds
`hi=0.35` and `lo=0.20`. Those numbers are not universal — in production the grader is an
LLM or a trained evaluator and the thresholds must be **calibrated on labelled data**.
The three-way branch is the transferable part.

In [ ]:
def crag_gate(query, k=3, hi=0.35, lo=0.20):
    """Return (action, retrieved_ids) with action in
    {"correct", "ambiguous", "incorrect"}."""
    hits = hybrid_rerank(query, k=k)
    ids = [d for d, _ in hits]
    best = max((s for _, s in hits), default=0.0)
    # TODO(you): best >= hi -> "correct"; best >= lo -> "ambiguous"; else "incorrect"
    action = "incorrect"
    return action, ids

for q in [QUERIES["exact-token"],
          QUERIES["single-fact"],
          "who owns the deployment schedule",
          "quarterly headcount planning"]:
    action, ids = crag_gate(q)
    print(f"{action:>10}  {ids[:3]}  <-  {q}")

In [ ]:
assert crag_gate(QUERIES["exact-token"])[0]        == "correct"
assert crag_gate(QUERIES["single-fact"])[0]        == "ambiguous"
assert crag_gate("quarterly headcount planning")[0] == "incorrect"
print("gate blocks generation on the out-of-corpus query instead of hallucinating")

<details>
<summary>Solution</summary>

```python
def crag_gate(query, k=3, hi=0.35, lo=0.20):
    hits = hybrid_rerank(query, k=k)
    ids = [d for d, _ in hits]
    best = max((s for _, s in hits), default=0.0)
    if best >= hi:
        action = "correct"
    elif best >= lo:
        action = "ambiguous"
    else:
        action = "incorrect"
    return action, ids
```

`"quarterly headcount planning"` shares no vocabulary with the corpus, so it scores 0.0 and
is gated off. Without the gate, those three irrelevant chunks would go straight into the
prompt and the model would answer from them — the exact silent failure CRAG exists to stop.

Two things to carry forward: (1) `"ambiguous"` is not a rounding error, it is the branch
where you *add* a second source rather than trusting or discarding; (2) CRAG wraps a frozen
model, which is why it is usually the practical choice over Self-RAG's fine-tuned
reflection tokens.
</details>

## 6. Key takeaways

- **RAG is a family, not an architecture.** The query shape — lookup, exact token,
  multi-hop, corpus-wide, aggregation — decides which member you need. The scoreboard
  above has no winning row.
- **Dense retrieval loses rare literals** because compression is what makes it semantic.
  BM25 recovers them; that is why hybrid exists, and we watched it fix `ORA-01555` at rank 1.
- **Contextual retrieval is nearly free** and recovered a chunk plain retrieval missed
  entirely — the work happens at indexing time, not query time.
- **Graph structure buys joins and synthesis** that top-$k$ cannot reach at any $k$, and
  the community report was *cheaper* in tokens than the chunks it replaced.
- **Agentic RAG matched GraphRAG on multi-hop with no special index**, at 3 LLM calls and
  ~3× the context. That trade is the decision, not a detail.
- **No text architecture can aggregate.** Route quantitative questions to SQL.
- **Evaluate per query shape.** A single averaged number would have shown five
  architectures performing "about the same" and hidden every finding above.

### Where to next

- [The lesson](https://ml-viz-ruby.vercel.app/courses/building-with-llms/14-rag-architectures) — the nine families, production evidence, and the decision procedure
- [Retrieval-Augmented Generation](https://ml-viz-ruby.vercel.app/courses/building-with-llms/04-retrieval-augmented-generation) — the base pipeline
- [Agents & Tool Use](https://ml-viz-ruby.vercel.app/courses/building-with-llms/05-agents-and-tool-use) — the loop agentic RAG runs inside
- [LLM Evaluation](https://ml-viz-ruby.vercel.app/courses/building-with-llms/08-llm-evaluation) — building the per-shape eval set
- [BM25: Lexical Ranking](https://ml-viz-ruby.vercel.app/wiki/bm25-ranking) · [Vector Databases](https://ml-viz-ruby.vercel.app/wiki/vector-databases)